## Author: Abdelaziz Neamatallah
### Date: 24.10.25
### Desc: This task should show the similarity between different datasets.

- The goal of this task is to analyze the similarity of the provided network traffic between the different files, in which it was collected. 
- So, we need to write a piece of code that satisfies the following requirements

1. For each of our datasets, we need to generate **traffic flows** from the corresponding network traffic.
    - Each flow should correspond to a time window of {2,4,6} mins and should only contain traffic exchanged between two communicating parties over the same highest-lecel protocol.
   - From the corresponding sequence of packets belonging to a given flow, we should extract only:
     - packet size
     - direction
     - inter-arrival packet time
   - All flows should have the same length.
   - And we should ensure that we do not lose information for longer flows.
   - Finally, we should differentiate between flows containing packets under attack, and those without attacks.

- Now lets try to understand some basic concepts before we proceed.
- Traffic flow mean: 
  - Think of the PCAP dataset as a long list of packets captured over time, which includes thousands of messages exchanged between many devices. 
  - A flow is a group of packets, that all belong to the same communication session between two hosts using the same protocol within a given time window.
  - in other words: all packets sent between host A and host B over the same protocol during X minutes. 
  - usually the traffic will looks like:

| Time (s) | Source       | Destination  | Protocol | Length | Attack? |
| -------- | ------------ | ------------ | -------- | ------ | ------- |
| 0.1      | 192.168.1.10 | 192.168.1.20 | S7comm   | 120    | No      |
| 0.2      | 192.168.1.20 | 192.168.1.10 | S7comm   | 130    | No      |
| 10.5     | 192.168.1.11 | 192.168.1.30 | HTTP     | 900    | Yes     |

  - So, you can see the first two packets are between the same pair using the same protocol, so they belong to one flow, and the last one is different so it belongs to another flow.
- Now, after we understood what does traffic flow means, we need o generate traffic flows for three durations:
  - 2 minutes
  - 4 minutes
  - 6 minutes
- this mean that we need to slice the whole capture timeline into chunks of 2,4, and 6 minutes each
  - 0:0 - 0:2 => flow window 1
  - 0:2 - 0:4 => flow window 2
- inside each window, we should group the packets by:
  - same source IP, destination IP, and protocol
  - each group will represent one flow.
- Then we should extract features from packets inside each flow
  - Packet size
  - Directions
  - Inter-arrival time (IAT): this is the time difference between consecutive packets in that flow, so each flow should look like a triplets.  
- It was mentioned that we need to diffrentiate between the packets marked as attacked, and other marked as normal.
- Also, all flows should have the same length. 

### Understanding of the datasets:
#### 2017QUT_S7Comm
- It consists of 3 sub-directories
  1. Graphs: 
  
| File / Folder              | Description                                                                                                                                                        |
| -------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| **`tshark_attacks`**       | Output of `tshark` (Wireshark CLI) during attack captures — lists packets, timestamps, fields (like function codes, packet sizes, etc.) for sessions under attack. |
| **`tshark_control`**       | Same, but for *normal (control)* traffic — used as baseline.                                                                                                       |
| **`gnuplot_graphs.gplot`** | A **Gnuplot** script that plots network statistics (e.g., number of S7 packets per second). Used to reproduce figures in their publication.                        |
| **`SCADA_traffic.ps`**     | PostScript plot (graph) showing SCADA traffic volume or timing patterns — likely an exported visualization.                                                        |

  2. LabeledDataset: => this is the core folder for us
     - It consists of 2 labeled datasets:
       - control_set: describes the regular operating procedure without incidents
       - s7_process_attacks: describes an operational sequence that is manipulated by an attacker 
  3. Logs:
     - Stores experiment log files from each participant system
- So, we can conclude that we will interact only with the LabeledDataset folder  

### Understanding the structure of s7_process_attacks:
- They captured network traffic from several network interfaces
  - Master: PLC or central controller
  - HMI: From the Human machine interface
  - Attacker: attacker traffic
- This contains one complete attack experiment, and these are the key files
    - master.pcap => these are the packets captured on the PLC or controller side, shows legitimate and injected commands received => this is the main dataset to analyze in our task
    - hmi.pcap => packets captured on the HMI (SCADA interface) side => also useful to be analyzed
    - attacker.pcap => Traffic seen by the attacking host, includes crafted malicious S7comm packets
- Then there is a folder called 20161215202830 => this looks like a date and time (2016/12/15 at 20:28:30 o'clock)
  - it contains subfolders each with a date then . the source of these data, and each folder contains some pcap files, following the same structure also
    - master
    - hmi
    - attacker
- I think, but I am not sure, that the attacker.pcap is a collection for all these sub-folders.pcap
- and same for hmi, and master. => Ask Dr. Asya

### Understanding of control_set
- This is much easier, it just contains three sub-files
  - Attacker.pcap
  - HMI.pcap
  - master.pcap

### Understanding of electra_S7comm dataset
- this is a .csv file, it is not pcap, should we handle it as is?
- [Dataset explaination](http://perception.inf.um.es/ICS-datasets/)

In [4]:
# read the electra_s7comm.csv file in pandas and print the first 5 rows
import pandas as pd 
df = pd.read_csv('../../Datasets/electra_s7comm/electra_s7comm.csv')
print(df.head())

MemoryError: Unable to allocate 5.77 GiB for an array with shape (387098466,) and data type complex128